# Economic Cycle Decomposition (1984-2024)

This notebook reproduces the pipeline:
- Build monthly MoM dataset (FRED, Stooq, Excel refs)
- Band-pass filtering at 200/100/42/21/12 months
- Hilbert phase, instantaneous period, outlier masking
- Plots and phase heatmaps

In [1]:
import os, pandas as pd, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from cycle_utils import bandpass_component, analytic_phase, instantaneous_period, flag_outliers

ROOT = Path('..').resolve()
df = pd.read_parquet(ROOT/'data/cycle_dataset_mom.parquet')
df.tail()

ModuleNotFoundError: No module named 'cycle_utils'

In [ ]:
TARGET_PERIODS = [200, 100, 42, 21, 12]
name = df.columns[0]
ts = df[name]
comps = {}
phases = {}
instTs = {}
for P in TARGET_PERIODS:
    y = bandpass_component(ts, P, bandwidth=0.25, order=4)
    ph = analytic_phase(y.fillna(0))
    inst = instantaneous_period(ph)
    ok = flag_outliers(inst, P, tol=0.35)
    comps[P] = y.where(ok)
    phases[P] = ph
    instTs[P] = inst
comp_df = pd.DataFrame(comps)
ph_df = pd.DataFrame(phases)
inst_df = pd.DataFrame(instTs)
comp_df.tail(), ph_df.tail(), inst_df.tail()

In [ ]:
fig, axes = plt.subplots(len(comp_df.columns)+1, 1, figsize=(12, 2.4*(len(comp_df.columns)+1)), sharex=True)
ts.plot(ax=axes[0], color='k', lw=1.0, label=name)
axes[0].legend(loc='upper left')
for i, col in enumerate(comp_df.columns, 1):
    comp_df[col].plot(ax=axes[i], lw=1.0, label=f'Band {col}m')
    axes[i].legend(loc='upper left')
plt.tight_layout()
plt.show()

In [2]:
sns.heatmap(ph_df.apply(np.degrees).transpose(), cmap='twilight', cbar_kws={'label':'phase (deg)'})
plt.title(f'Phase heatmap: {name}')
plt.show()

NameError: name 'ph_df' is not defined